# 01. Данные
Структура данных X5 RetailHero: объёмы, столбцы, устройство чеков.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
DATA = 'data/data/'
print('duckdb', duckdb.__version__)

duckdb 2.0.0


In [2]:
for f in ['clients', 'products', 'purchases', 'uplift_train']:
    path = f"{DATA}{f}.csv"
    n = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
    cols = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").fetchall()]
    print(f"{f}: {n:,} строк")
    print("   ", cols, "\n")

clients: 400,162 строк
    ['client_id', 'first_issue_date', 'first_redeem_date', 'age', 'gender'] 

products: 43,038 строк
    ['product_id', 'level_1', 'level_2', 'level_3', 'level_4', 'segment_id', 'brand_id', 'vendor_id', 'netto', 'is_own_trademark', 'is_alcohol'] 

purchases: 45,786,568 строк
    ['client_id', 'transaction_id', 'transaction_datetime', 'regular_points_received', 'express_points_received', 'regular_points_spent', 'express_points_spent', 'purchase_sum', 'store_id', 'product_id', 'product_quantity', 'trn_sum_from_iss', 'trn_sum_from_red'] 

uplift_train: 200,039 строк
    ['client_id', 'treatment_flg', 'target'] 



## Как устроены чеки
Проверяем: за какой период история покупок (нет ли утечки данных
после рассылки), одна строка — это чек или позиция, и как хранится сумма чека.

In [3]:
P = f"read_csv_auto('{DATA}purchases.csv')"

# За какой период история покупок
print(con.execute(f"""
    SELECT MIN(transaction_datetime) AS first_purchase,
           MAX(transaction_datetime) AS last_purchase
    FROM {P}
""").fetchall())

# что значит одна строка
print(con.execute(f"""
    SELECT COUNT(*)                       AS rows,
           COUNT(DISTINCT transaction_id) AS transactions
    FROM {P}
""").fetchall())

# Повторяется ли сумма чека в каждой строке чека?
print(con.execute(f"""
    SELECT COUNT(*) AS transactions_with_different_sums
    FROM (
        SELECT transaction_id
        FROM {P}
        GROUP BY transaction_id
        HAVING COUNT(DISTINCT purchase_sum) > 1
    )
""").fetchall())

[(datetime.datetime(2018, 11, 21, 21, 2, 33), datetime.datetime(2019, 3, 18, 23, 40, 3))]
[(45786568, 8045201)]
[(28,)]


- Период: 21.11.2018 – 18.03.2019
- Одна строка = позиция в чеке (45,8 млн строк, 8 млн чеков)
- purchase_sum повторяется в каждой позиции → средний чек считать по уникальным чекам